In [1]:
import os
from typing import Optional
import numpy as np
import pandas as pd
from openpyxl import load_workbook
import re


def assign_offense_defense(team, opponent, plays):
    """
    Add 'offense' and 'defense' columns to the plays DataFrame based on
    'ODK' (and, for ODK == 'K', on which team was on offense immediately
    before the kicking play — not a fixed team/opponent mapping, since
    either team can punt, kick off, attempt a FG, etc.).

    Note the very first row of a game, if it's itself a 'K' row (e.g. the
    opening kickoff), has no previous play to inherit from, so offense
    and defense come out None for that one row. That has to be seeded
    manually (who received the opening kickoff) if it matters for your
    use case — it can't be inferred from the data alone.
    """
    df = plays.copy()
    df = df[df["ODK"] != "S"].reset_index(drop=True)  # drop rows that were throwing stuff off

    # Kicking plays where the team that WAS on offense stays the offense
    # (it's still fundamentally their play, made or not). "Punt Rec" is
    # included here, NOT in the flip set below: its DN/DIST always
    # continues the kicking team's stalled drive (e.g. DN=4, DIST=17
    # matching their prior 3rd-and-13), so it has to stay attributed to
    # the team that was actually driving, not the team about to receive.
    SAME_AS_PREV_OFFENSE = {
        "Punt", "Punt Rec", "FG", "FG Block", "Extra Pt.", "Extra Pt. Block",
        "2 Pt.", "2 Pt. Block", "KO Rec",
    }
    # Kicking plays where offense flips to the OTHER team.
    FLIP_FROM_PREV_OFFENSE = {"KO"}

    offense_col, defense_col = [], []
    prev_offense, prev_defense = None, None

    for _, row in df.iterrows():
        odk = row["ODK"]
        if odk == "O":
            off, defn = team, opponent
        elif odk == "D":
            off, defn = opponent, team
        elif odk == "K":
            pt = row["PLAY TYPE"]
            if pt in SAME_AS_PREV_OFFENSE:
                off, defn = prev_offense, prev_defense
            elif pt in FLIP_FROM_PREV_OFFENSE:
                off, defn = prev_defense, prev_offense
            else:
                off, defn = None, None  # unmatched PLAY TYPE for a 'K' row
        else:
            off, defn = None, None

        offense_col.append(off)
        defense_col.append(defn)
        if off is not None:
            prev_offense, prev_defense = off, defn

    df["offense"] = offense_col
    df["defense"] = defense_col

    return df

def add_success_metrics(plays):
    """
    Add 'success', 'chunk_play', and 'explosive_play' columns to a plays DataFrame.

    Parameters
    ----------
    plays : pd.DataFrame
        Must contain columns 'DN', 'DIST', 'GN/LS', and 'ODK'.

    Returns
    -------
    pd.DataFrame
        A copy of `plays` with the three new columns added.
    """
    df = plays.copy()

    def _get_success(row):
        if row["ODK"] == "K":
            return None

        dn = row["DN"]
        dist = row["DIST"]
        gain = row["GN/LS"]

        if pd.isna(dn) or pd.isna(dist) or pd.isna(gain):
            return None

        if dn == 1:
            if dist == 0:
                return None
            return 1 if gain >= 0.40 * dist else 0
        elif dn == 2:
            if dist == 0:
                return None
            return 1 if gain >= 0.60 * dist else 0
        elif dn in (3, 4):
            return 1 if gain >= dist else 0
        else:
            return None  # unexpected DN value

    df["success"] = df.apply(_get_success, axis=1)

    is_kick = df["ODK"] == "K"

    df["chunk_play"] = np.where(
        is_kick,
        None,
        np.where((df["GN/LS"] >= 10) & (df["GN/LS"] < 20), 1, 0)
    )

    df["explosive_play"] = np.where(
        is_kick,
        None,
        np.where(df["GN/LS"] >= 20, 1, 0)
    )

    return df

# Kicking plays that don't represent a live down/distance situation, so no
# ep is computed for them. Anything else with ODK == "K" (punts, field
# goals, blocked kicks, etc.) still has a real down/distance/field-position
# context and gets an ep value.
NON_SCRIMMAGE_KICK_TYPES = {"KO", "KO Rec", "Extra Pt.", "Extra Pt. Block"}


def add_score_columns(plays, team, opponent):
    """
    Add cumulative 'team_score' and 'opponent_score' columns, plus
    'offense_score' and 'defense_score', to the plays DataFrame by
    processing plays in order of 'PLAY #'.

    'offense_score' / 'defense_score' just mirror 'team_score' /
    'opponent_score' depending on which side is listed in 'offense' for
    that row, so the same play's score can be read from either team's
    perspective without a lookup elsewhere.

    Parameters
    ----------
    plays : pd.DataFrame
        Must contain columns 'PLAY #', 'RESULT', 'PLAY TYPE', 'offense', 'defense'.
    team : str
    opponent : str

    Returns
    -------
    pd.DataFrame
        A copy of `plays`, sorted by 'PLAY #' ascending, with 'team_score',
        'opponent_score', 'offense_score', and 'defense_score' columns
        added (running totals as of each play).
    """
    df = plays.sort_values("PLAY #", ascending=True).reset_index(drop=True)

    team_score = 0
    opponent_score = 0

    team_scores_col = []
    opponent_scores_col = []
    offense_scores_col = []
    defense_scores_col = []

    for _, row in df.iterrows():
        result = row["RESULT"] if pd.notna(row["RESULT"]) else ""
        play_type = row["PLAY TYPE"]
        offense = row["offense"]
        defense = row["defense"]

        # Defensive touchdown (pick-six, fumble return, etc.)
        if "Def TD" in result:
            if defense == team:
                team_score += 6
            elif defense == opponent:
                opponent_score += 6

        # Offensive touchdown (only if it's not a "Def TD")
        elif "TD" in result:
            if offense == team:
                team_score += 6
            elif offense == opponent:
                opponent_score += 6

        # Extra point
        if play_type == "Extra Pt.":
            if result == "Good":
                if offense == team:
                    team_score += 1
                elif offense == opponent:
                    opponent_score += 1

        # Field goal
        if play_type == "FG":
            if result == "Good":
                if offense == team:
                    team_score += 3
                elif offense == opponent:
                    opponent_score += 3

        # Blocked FG returned/recovered for score (e.g., "FG Block" credited as good)
        if play_type == "FG Block":
            if result == "Good":
                if offense == team:
                    team_score += 3
                elif offense == opponent:
                    opponent_score += 3

        # Blocked Extra Pt. returned/recovered for score
        if play_type == "Extra Pt. Block":
            if result == "Good":
                if offense == team:
                    team_score += 1
                elif offense == opponent:
                    opponent_score += 1

        # Safety
        if "Safety" in result:
            if defense == team:
                team_score += 2
            elif defense == opponent:
                opponent_score += 2

        team_scores_col.append(team_score)
        opponent_scores_col.append(opponent_score)

        # offense_score/defense_score just mirror whichever side is on
        # offense for this row, using the running totals above.
        if offense == team:
            offense_scores_col.append(team_score)
            defense_scores_col.append(opponent_score)
        elif offense == opponent:
            offense_scores_col.append(opponent_score)
            defense_scores_col.append(team_score)
        else:
            offense_scores_col.append(np.nan)
            defense_scores_col.append(np.nan)

    df["team_score"] = team_scores_col
    df["opponent_score"] = opponent_scores_col
    df["offense_score"] = offense_scores_col
    df["defense_score"] = defense_scores_col

    return df


def calculate_ep_epa(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add YARDLINE_100, ep, and epa columns to a dataframe of high school
    football plays.

    Expected columns on input (run add_score_columns first to get
    offense_score/defense_score):
        PLAY #         - play order within the game
        QTR            - quarter, used for the Q4 blowout dampener
        ODK            - "O", "D", or "K" (offense / defense / kicking play)
        PLAY TYPE      - e.g. "Run", "Pass", "Punt", "FG", "KO", ...
        DN             - down (1-4)
        DIST           - yards to go for a first down
        YARD LN        - raw yard line, positive on own side / negative on
                         opponent's side, per your data convention
        offense_score  - current offense's running score (see add_score_columns)
        defense_score  - current defense's running score (see add_score_columns)
        offense        - which team is on offense this play (used to
                         detect possession changes for epa)
        RESULT         - play result text (used for scoring detection
                         and the interception/fumble/turnover flags)

    ep is left as NaN only for kicking plays that don't have a real
    down/distance situation: extra points and blocked extra points.
    Punts, field goals, and blocked field goals still get an ep computed,
    since those are the "what happens on 4th down" plays. Kickoffs (PLAY
    TYPE == "KO" or "KO Rec") get a fixed ep instead of NaN, equal to
    1st & 10 at the -25 (yardline_100 = 75) — the receiving team's
    typical starting field position.

    A Q4/blowout dampener is applied: if it's the 4th quarter and the
    score margin is more than 21, ep is scaled down (garbage-time plays
    swing the scoreboard less than the formula would otherwise suggest).

    The -2 floor is skipped for punts specifically (PLAY TYPE == "Punt"
    or "Punt Rec"), so a punt's pre-kick ep can go below -2 rather than
    clipping every bad 4th-and-long situation to the same value — this
    keeps punt epa reflecting the actual field-position swing instead of
    the clip.

    epa is calculated one of two ways, in PLAY # order:
      - If the play ends in a score (offensive TD, defensive TD, safety,
        or made field goal / FG block), epa = the point value of that
        score minus this play's ep. This avoids relying on the following
        kickoff or extra point, which have no ep of their own.
      - Otherwise, epa = the next play's ep minus this play's ep, with
        the next play's ep NEGATED first if possession changed hands
        (offense differs between this row and the next row) — covering
        interceptions, lost fumbles, turnovers on downs, and punts alike,
        without needing to check RESULT for each case individually.
    epa is left as NaN when this play's ep is NaN, when it's the last
    play in the dataframe, or when the next play's ep is NaN (e.g. end
    of half/game, or the next play is itself a kickoff/PAT).

    interception / fumble / turnover are binary (1/0) flags read off the
    RESULT column: interception = 1 if RESULT contains "Interception",
    fumble = 1 if RESULT contains "Fumble", and turnover = 1 if either
    of those is 1 (e.g. a fumble that's recovered by the offense still
    sets fumble = 1 and turnover = 1, since it doesn't distinguish a lost
    fumble from one the offense kept — narrow the RESULT match if you
    need to exclude recovered fumbles).
    """
    df = df.copy()

    # --- YARDLINE_100: distance from the opponent's goal line, 1-99 ---
    df["YARDLINE_100"] = np.where(
        df["YARD LN"] > 0,
        df["YARD LN"],
        100 + df["YARD LN"],
    )

    # --- interception / fumble / turnover flags, from RESULT ---
    result_text = df["RESULT"].fillna("")
    df["interception"] = result_text.str.contains("Interception").astype(int)
    df["fumble"] = result_text.str.contains("Fumble").astype(int)
    df["turnover"] = ((df["interception"] == 1) | (df["fumble"] == 1)).astype(int)

    # --- EP formula pieces, calibrated against cfbfastR's published
    # EP-by-field-position-and-down table (distance = 10, tied game).
    # Field curve fixed at a -2 floor / 7 ceiling so it stays interpretable.
    def ep_field(yardline_100):
        return -2 + 9 / (1 + np.exp(0.037 * (yardline_100 - 50)))

    def down_penalty(down, dist, yardline_100):
        # Extra cost for downs beyond 1st, fit per-down against cfbfastR's
        # actual gaps rather than assumed as one flat per-down constant.
        # 2nd down is close to a flat tax; 3rd/4th taper off near the goal
        # line since field-goal range still has value even on a bad down.
        if down == 1:
            base = 0.0
        elif down == 2:
            base = 0.62
        elif down == 3:
            base = 1.12 + 0.0184 * yardline_100 - 0.00016 * yardline_100 ** 2
        else:
            base = 1.62 + 0.0735 * yardline_100 - 0.00075 * yardline_100 ** 2

        # cfbfastR's public table only covers distance = 10, so this part
        # is still a rough estimate, not something fit to real data.
        extra_distance = 0.06 * (dist - 10)
        return base + extra_distance

    # Kickoff plays (both "KO" and "KO Rec") get a fixed ep equal to 1st &
    # 10 at the -25 (i.e. the receiving team's typical starting field
    # position), rather than NaN.
    KICKOFF_YARDLINE_100 = 75  # -25 under this dataframe's YARD LN convention

    def compute_ep(row):
        if row["PLAY TYPE"] in ("KO", "KO Rec"):
            raw_ep = ep_field(KICKOFF_YARDLINE_100) - down_penalty(
                1, 10, KICKOFF_YARDLINE_100
            )
            if row["QTR"] == 4 and abs(row["offense_score"] - row["defense_score"]) > 21:
                raw_ep *= 0.6
            return float(np.clip(raw_ep, -2, 7))

        if row["ODK"] == "K" and row["PLAY TYPE"] in NON_SCRIMMAGE_KICK_TYPES:
            return np.nan

        raw_ep = ep_field(row["YARDLINE_100"]) - down_penalty(
            row["DN"], row["DIST"], row["YARDLINE_100"]
        )

        if row["QTR"] == 4 and abs(row["offense_score"] - row["defense_score"]) > 21:
            raw_ep *= 0.6

        # Punts are usually attempted from already-bad situations (4th &
        # long), which would otherwise all get flattened to the same -2
        # floor. Skipping the floor here lets a punt's pre-kick ep still
        # distinguish a merely-bad spot from a truly awful one, so the
        # resulting epa reflects the actual punt rather than the clip.
        lower_bound = -np.inf if row["PLAY TYPE"] in ("Punt", "Punt Rec") else -2
        return float(np.clip(raw_ep, lower_bound, 7))

    df["ep"] = df.apply(compute_ep, axis=1)
    df = df.sort_values("PLAY #").reset_index(drop=True)

    # --- score_value: point value of a play that ends the drive, signed
    # from that play's own offense's perspective. None if the play
    # doesn't end in a score. ---
    def score_value(row):
        result = row["RESULT"] if pd.notna(row["RESULT"]) else ""
        play_type = row["PLAY TYPE"]

        if "Def TD" in result:
            return -7.0
        elif "TD" in result:
            return 7.0
        elif "Safety" in result:
            return -2.0
        elif play_type in ("FG", "FG Block") and result == "Good":
            return 3.0
        return None

    # --- EPA ---
    # Scoring plays: epa = point value of the score - ep (never looks at
    # the following kickoff/PAT row, which has no ep).
    # Non-scoring plays: epa = next play's ep - this play's ep, with the
    # next play's ep NEGATED if possession changed hands (interception,
    # lost fumble, turnover on downs, punt, etc.) so a turnover reads as
    # a loss of equity for the team that had the ball, not a gain.
    ep_vals = df["ep"].to_numpy()
    offense_vals = df["offense"].to_numpy()
    n = len(df)
    epa_vals = np.full(n, np.nan)

    for i in range(n):
        if pd.isna(ep_vals[i]):
            continue  # no ep for this play (kickoff/PAT) -> epa stays NaN

        sv = score_value(df.iloc[i])
        if sv is not None:
            epa_vals[i] = sv - ep_vals[i]
            continue

        if i == n - 1 or pd.isna(ep_vals[i + 1]):
            continue  # last play, or next play has no ep -> leave as NaN

        possession_changed = offense_vals[i] != offense_vals[i + 1]
        next_ep = -ep_vals[i + 1] if possession_changed else ep_vals[i + 1]
        epa_vals[i] = next_ep - ep_vals[i]

    df["epa"] = epa_vals

    return df


# PLAY TYPE values that count as each attempt/flag column below.
KICKOFF_TYPES = {"KO", "KO Rec"}
PUNT_TYPES = {"Punt", "Punt Rec"}
XP_TYPES = {"Extra Pt.", "Extra Pt. Block"}
FG_TYPES = {"FG", "FG Block"}


def add_play_detail_columns(df: pd.DataFrame, team: str, opponent: str, date: str, week) -> pd.DataFrame:
    """
    Add play-type flags, game_id, drive, and drive_result columns to a
    plays dataframe.

    Run this AFTER add_score_columns (needs offense_score/defense_score)
    and calculate_ep_epa (needs the turnover column). Also assumes a
    'success' column (used for THIRD_DOWN_CONVERTED, FOURTH_DOWN_CONVERTED,
    and the "Turnover on Downs" drive_result) and a 'GN/LS' column (used
    for TACKLE_FOR_LOSS) already exist on the input.

    Columns added
    -------------
    KICKOFF, PUNT, XP_ATTEMPT, FG_ATTEMPT : 1/0, from PLAY TYPE
    KICK_MADE      : 1 if an XP/FG attempt and RESULT == "Good"
    TOUCHBACK      : 1 if RESULT == "Touchback"
    QB_SCRAMBLE    : 1 if RESULT == "Scramble"
    TACKLE_FOR_LOSS: 1 if RESULT == "Rush" and GN/LS < 0
    SACK           : 1 if RESULT contains "Sack"
    PENALTY        : 1 if RESULT == "Penalty"
    SCORE_DIFFERENTIAL : offense_score - defense_score
    RUSH, PASS     : 1 if PLAY TYPE contains "Run" / "Pass"
    THIRD_DOWN_CONVERTED, FOURTH_DOWN_CONVERTED : 1 if DN == 3/4 and success == 1
    game_id        : f"{team}_{opponent}_{date}", same for every row
    WEEK           : the 'week' argument, same for every row
    drive          : running count, incrementing each time 'offense' changes
    YDS_NET        : running total of GN/LS within the current drive,
                     restarting at 0 on the first play of each new drive
    SERIES         : like drive, but also increments on every 1st down
                     where the offense is unchanged from the previous
                     play (including back-to-back 1st-and-10s, e.g. from
                     a penalty replaying the down), not just on a change
                     of possession. Kickoffs are NaN (not part of any
                     series), and extra points carry over the same
                     SERIES value as the play before them.
    drive_result   : one of "Touchdown", "FG Attempt", "Punt", "Turnover",
                     "Safety", "Turnover on Downs", "End of Half",
                     "End of Game", or "Error" (if the last play of the
                     drive doesn't match any of those conditions) —
                     determined from the last play of each drive and
                     applied to every play in that drive. "Touchdown" is
                     triggered by PLAY TYPE containing "Extra Pt." or "2 Pt."
    SERIES_RESULT  : one of "First Down", "Touchdown", "Punt", "FG Made",
                     "FG Miss", "Turnover", "Safety", "Turnover on Downs",
                     "End of Half", "End of Game", "Error" (if the last
                     play of the series doesn't match any of those
                     conditions), or NaN (for kickoffs, which have no
                     SERIES) — determined from the last play of each
                     series and applied to every play in that series.
                     "Touchdown" is triggered by PLAY TYPE containing
                     "Extra Pt." or "2 Pt."
    SERIES_SUCCESS : 1 if SERIES_RESULT is "Touchdown", "First Down", or
                     "FG Made", 0 if SERIES_RESULT is any other non-NaN
                     value (including "Error"), and NaN if SERIES_RESULT
                     itself is NaN (kickoffs)
    """
    df = df.copy()
    df = df.sort_values("PLAY #").reset_index(drop=True)

    play_type = df["PLAY TYPE"]
    result = df["RESULT"].fillna("")

    df["KICKOFF"] = play_type.isin(KICKOFF_TYPES).astype(int)
    df["PUNT"] = play_type.isin(PUNT_TYPES).astype(int)
    df["XP_ATTEMPT"] = play_type.isin(XP_TYPES).astype(int)
    df["FG_ATTEMPT"] = play_type.isin(FG_TYPES).astype(int)

    df["KICK_MADE"] = (
        play_type.isin(XP_TYPES | FG_TYPES) & (df["RESULT"] == "Good")
    ).astype(int)

    df["TOUCHBACK"] = (df["RESULT"] == "Touchback").astype(int)
    df["QB_SCRAMBLE"] = (df["RESULT"] == "Scramble").astype(int)
    df["TACKLE_FOR_LOSS"] = (
        (df["RESULT"] == "Rush") & (df["GN/LS"] < 0)
    ).astype(int)
    df["SACK"] = result.str.contains("Sack").astype(int)
    df["PENALTY"] = (df["RESULT"] == "Penalty").astype(int)

    df["SCORE_DIFFERENTIAL"] = df["offense_score"] - df["defense_score"]

    df["RUSH"] = play_type.str.contains("Run", na=False).astype(int)
    df["PASS"] = play_type.str.contains("Pass", na=False).astype(int)

    df["THIRD_DOWN_CONVERTED"] = (
        (df["DN"] == 3) & (df["success"] == 1)
    ).astype(int)
    df["FOURTH_DOWN_CONVERTED"] = (
        (df["DN"] == 4) & (df["success"] == 1)
    ).astype(int)

    df["game_id"] = f"{team}_{opponent}_{date}"
    df["WEEK"] = week

    # --- drive: increments every time 'offense' changes from the previous
    # play, OR at halftime (Q2 -> Q3) even if offense happens to be the
    # same team on both sides of the gap (e.g. a deferred-receive team
    # also gets the ball right before half and right after it) — without
    # this, that gap wouldn't register as a new drive at all. ---
    halftime_boundary = (df["QTR"] == 3) & (df["QTR"].shift() == 2)
    df["drive"] = ((df["offense"] != df["offense"].shift()) | halftime_boundary).cumsum()

    # --- YDS_NET: running total of GN/LS within each drive, restarting at
    # the start of every new drive. Missing GN/LS values are treated as 0
    # so they don't break the running total for the rest of the drive. ---
    df["YDS_NET"] = df.groupby("drive")["GN/LS"].transform(
        lambda s: s.fillna(0).cumsum()
    )

    # --- SERIES: like drive, but also increments on every 1st down where
    # the offense is unchanged from the previous play (including
    # back-to-back 1st-and-10s), and also forces a break at halftime for
    # the same reason 'drive' does above. Kickoffs and extra points don't
    # represent a real down/series, so they're excluded from the
    # increment logic below: kickoffs end up NaN, extra points just carry
    # over whatever series the play before them was in.
    normal_play_mask = ~df["PLAY TYPE"].isin(NON_SCRIMMAGE_KICK_TYPES)
    normal_plays = df.loc[normal_play_mask]

    new_series = (
        (normal_plays["offense"] != normal_plays["offense"].shift())
        | (
            (normal_plays["DN"] == 1)
            & (normal_plays["offense"] == normal_plays["offense"].shift())
        )
        | ((normal_plays["QTR"] == 3) & (normal_plays["QTR"].shift() == 2))
    )
    series_for_normal_plays = new_series.cumsum()

    df["SERIES"] = np.nan
    df.loc[normal_play_mask, "SERIES"] = series_for_normal_plays.to_numpy()
    df["SERIES"] = df["SERIES"].ffill()  # carries into KO/XP rows for now
    df.loc[df["PLAY TYPE"].isin(KICKOFF_TYPES), "SERIES"] = np.nan  # then null kickoffs back out

    # --- drive_result: computed on the last play of each drive, then
    # broadcast to every play in that drive ---
    is_last_of_drive = df["drive"] != df["drive"].shift(-1)
    n = len(df)

    def last_play_result(i):
        pt = str(df.at[i, "PLAY TYPE"])
        res = df.at[i, "RESULT"] if pd.notna(df.at[i, "RESULT"]) else ""

        if pt in ("KO", "KO Rec"):
            # A "drive" whose only/last play is a bare kickoff isn't a
            # real possession to classify (e.g. the opening kickoff of a
            # game or a half) — leave it null rather than forcing it
            # through the categories below.
            return None

        if "Extra Pt." in pt or "2 Pt." in pt:
            return "Touchdown"
        elif "FG" in pt:
            return "FG Attempt"
        elif "Punt" in pt:
            return "Punt"
        elif df.at[i, "turnover"] == 1:
            return "Turnover"
        elif "Safety" in res:
            return "Safety"
        elif df.at[i, "DN"] == 4 and df.at[i, "success"] == 0:
            return "Turnover on Downs"
        elif df.at[i, "QTR"] == 2 and i + 1 < n and df.at[i + 1, "QTR"] == 3:
            return "End of Half"
        elif df.at[i, "QTR"] == 4 and i == n - 1:
            return "End of Game"
        return "Error"

    drive_result_raw = pd.Series(np.nan, index=df.index, dtype=object)
    for i in df.index[is_last_of_drive]:
        drive_result_raw.at[i] = last_play_result(i)

    df["drive_result"] = drive_result_raw.groupby(df["drive"]).transform("last")

    # --- SERIES_RESULT: computed on the last play of each series, then
    # broadcast to every play in that series. Rows with SERIES == NaN
    # (kickoffs) end up NaN here too, since they aren't part of a series.
    is_last_of_series = (df["SERIES"] != df["SERIES"].shift(-1)) & df["SERIES"].notna()

    def last_play_series_result(i):
        pt = str(df.at[i, "PLAY TYPE"])
        res = df.at[i, "RESULT"] if pd.notna(df.at[i, "RESULT"]) else ""

        if pt in ("KO", "KO Rec"):
            return None

        if (
            i + 1 < n
            and df.at[i + 1, "offense"] == df.at[i, "offense"]
            and df.at[i + 1, "DN"] == 1
        ):
            return "First Down"
        elif "Extra Pt." in pt or "2 Pt." in pt:
            return "Touchdown"
        elif df.at[i, "PUNT"] == 1:
            return "Punt"
        elif df.at[i, "FG_ATTEMPT"] == 1 and df.at[i, "KICK_MADE"] == 1:
            return "FG Made"
        elif df.at[i, "FG_ATTEMPT"] == 1 and df.at[i, "KICK_MADE"] == 0:
            return "FG Miss"
        elif df.at[i, "turnover"] == 1:
            return "Turnover"
        elif res == "Safety":
            return "Safety"
        elif df.at[i, "DN"] == 4 and df.at[i, "success"] == 0:
            return "Turnover on Downs"
        elif df.at[i, "QTR"] == 2 and i + 1 < n and df.at[i + 1, "QTR"] == 3:
            return "End of Half"
        elif df.at[i, "QTR"] == 4 and i == n - 1:
            return "End of Game"
        return "Error"

    series_result_raw = pd.Series(np.nan, index=df.index, dtype=object)
    for i in df.index[is_last_of_series]:
        series_result_raw.at[i] = last_play_series_result(i)

    df["SERIES_RESULT"] = series_result_raw.groupby(df["SERIES"]).transform("last")
    df["SERIES_SUCCESS"] = np.where(
        df["SERIES_RESULT"].isna(),
        np.nan,
        df["SERIES_RESULT"].isin(["Touchdown", "First Down", "FG Made"]).astype(float),
    )

    return df


def add_dowling_offense_plays(plays: pd.DataFrame, offense_plays: pd.DataFrame) -> pd.DataFrame:
    """
    Return a copy of `plays` where, for every row with offense ==
    "Dowling Catholic", the OFF FORM and OFF PLAY values are replaced
    with the corresponding values from `offense_plays`, matched on
    PLAY #.

    `plays` and `offense_plays` are expected to have identical columns
    and rows, differing only in OFF FORM / OFF PLAY.
    """
    df = plays.copy()
    lookup = offense_plays.set_index("PLAY #")[["OFF FORM", "OFF PLAY"]]

    is_dowling = df["offense"] == "Dowling Catholic"
    play_nums = df.loc[is_dowling, "PLAY #"]

    df.loc[is_dowling, "OFF FORM"] = play_nums.map(lookup["OFF FORM"])
    df.loc[is_dowling, "OFF PLAY"] = play_nums.map(lookup["OFF PLAY"])

    schemes = [
        "HERKY", "STORM", "BULLDOGS", "PANTHERS", "CYCLONES", "HAWKEYES",
        "PATRIOTS", "DRAW", "SUPERSONICS", "MONEY", "SEATTLE", "JAYHAWKS",
        "IZZY", "MONSTER", "BLUNT"
    ]

    pattern = r"\b(" + "|".join(map(re.escape, schemes)) + r")\b"

    df["RUN SCHEME"] = (
        df["OFF PLAY"]
        .astype("string")
        .str.extract(pattern, flags=re.IGNORECASE, expand=False)
        .str.upper()
    )

    return df

def curate_play_by_play_data(plays: pd.DataFrame, team: str, opponent: str, date: str, week,):
    plays = assign_offense_defense(team=team, opponent=opponent, plays=plays)
    plays = add_success_metrics(plays = plays)
    plays = add_score_columns(plays=plays, team=team, opponent=opponent)
    plays = calculate_ep_epa(df=plays)
    plays = add_play_detail_columns(df=plays, team=team, opponent=opponent, date=date, week=week)
    return plays

def curate_pbp_write_to_excel(
    plays: pd.DataFrame,
    team: str,
    opponent: str,
    date: str,
    week,
    dowling_film: bool = False,
    offense_plays: Optional[pd.DataFrame] = None,
):
    """
    Curate a game's play-by-play data and write it to curated-pbp.xlsx,
    appending to "Sheet1" if the file already exists, or creating it
    (with a header row) if it doesn't.
    """
    plays = curate_play_by_play_data(
        plays=plays, team=team, opponent=opponent, date=date, week=week
    )

    if dowling_film:
        offense_plays = curate_play_by_play_data(
            plays=offense_plays, team=team, opponent=opponent, date=date, week=week
        )
        plays = add_dowling_offense_plays(plays, offense_plays)

    file_path = "curated-pbp.xlsx"
    sheet_name = "Sheet1"

    if not os.path.exists(file_path):
        # File doesn't exist yet: create it fresh, with a header row.
        with pd.ExcelWriter(file_path, engine="openpyxl", mode="w") as writer:
            plays.to_excel(writer, sheet_name=sheet_name, index=False, header=True)
        print("Data written successfully!")
        return

    # File exists: find where to append, and whether a header is needed
    # (only if the sheet is otherwise empty).
    wb = load_workbook(file_path)
    start_row = wb[sheet_name].max_row if sheet_name in wb.sheetnames else 0
    wb.close()

    header = start_row == 0

    with pd.ExcelWriter(file_path, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
        plays.to_excel(
            writer, sheet_name=sheet_name, startrow=start_row, index=False, header=header
        )
    print("Data written successfully!")

In [2]:
# Week 1 Valley Data
plays = pd.read_excel('Dowling/dowling-vs-valley-082826.xlsx')
offense_plays = pd.read_excel('Dowling/dowling-vs-valley-offense.xlsx')
team = "Dowling Catholic"
opponent = "Valley"
date = "2026_08_28"
week = 1

# Read, edit, and save off Valley data
plays = pd.read_excel('Dowling/dowling-vs-valley-082826.xlsx')
plays = plays[plays['ODK'] != "S"]
plays = plays[plays['PLAY #'] != 82]
plays = plays.sort_values("PLAY #").reset_index(drop=True)
plays["PLAY #"] = range(1, len(plays) + 1)

offense_plays = pd.read_excel('Dowling/dowling-vs-valley-offense.xlsx')
offense_plays = offense_plays[offense_plays['PLAY #'] != 57]
offense_plays = offense_plays.sort_values("PLAY #").reset_index(drop=True)
offense_plays["PLAY #"] = range(1, len(offense_plays) + 1)

curate_pbp_write_to_excel(plays=plays, team=team, opponent=opponent, date=date, week=week, dowling_film=True, offense_plays=offense_plays)

Data written successfully!


In [3]:
# Week 2 Johnston
plays = pd.read_excel('Dowling/dowling-vs-johnston-090426.xlsx')
offense_plays = pd.read_excel('Dowling/dowling-vs-johnston-offense.xlsx')
team = "Dowling Catholic"
opponent = "Johnston"
date = "2026_09_04"
week = 2

curate_pbp_write_to_excel(plays=plays, team=team, opponent=opponent, date=date, week=week, dowling_film=True, offense_plays=offense_plays)

Data written successfully!


In [4]:
# Week 3 DCG
plays = pd.read_excel('Dowling/dowling-vs-dcg-091126.xlsx')
offense_plays = pd.read_excel('Dowling/dowling-vs-dcg-offense.xlsx')
team = "Dowling Catholic"
opponent = "DCG"
date = "2026_09_11"
week = 3

curate_pbp_write_to_excel(plays=plays, team=team, opponent=opponent, date=date, week=week, dowling_film=True, offense_plays=offense_plays)

Data written successfully!


In [5]:
# Week 4 Southeast Polk
plays = pd.read_excel('Dowling/dowling-vs-sep-091826.xlsx')
offense_plays = pd.read_excel('Dowling/dowling-vs-sep-offense.xlsx')
team = "Dowling Catholic"
opponent = "SEP"
date = "2026_09_18"
week = 4

curate_pbp_write_to_excel(plays=plays, team=team, opponent=opponent, date=date, week=week, dowling_film=True, offense_plays=offense_plays)

<positron-console-cell-5>:687: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['MIRROR STORM' 'PATRIOTS' 'BREAK SPRAY JOHN C' 'BLUNT KEY 2'
 'MIRROR STORM' 'BLUNT KEY 2' 'BOUND SWAP STICK' 'BOUND FEVER'
 'MIRROR BULLDOGS KEY 2' 'MONEY' nan 'PANTHERS' 'PATRIOTS KEY 2'
 'TEAR ICE COVER' nan 'PATRIOTS KEY 3' 'PIPE JUKE' 'MIRROR STORM' nan
 'PATRIOTS' 'PATRIOTS' 'PATRIOTS' 'SNAP BEAU POST' 'STORM' 'BAD SNAP'
 'BOUND UNCOIL DIVA SWITCH' nan 'PATRIOTS' 'HAWKEYES NOAH' 'BAD SNAP' nan
 'BOUND COVER LOW' 'ICE ALABAMA' 'MONSTER' 'STORM' 'BREAK SWAP STICK'
 'FLASH LIZARD' 'BOUND COVER' 'MIRROR STORM' 'BOUND UNCOIL COVER' nan nan
 'PATRIOTS C' 'PATRIOTS C' 'BREAK SPRAY JOHN C' 'BOUND UNCOIL DIVA SWITCH'
 'PATRIOTS' 'MIRROR STORM' 'PATRIOTS' 'BOUND PIPE DIVA SWITCH'
 'MIRROR STORM' 'MIRROR BULLDOGS SLICE EGYPT' nan nan 'MIRROR STORM'
 'PATRIOTS' 'PATRIOTS' nan 'BOUND NILE' 'PATRIOTS' 'BREAK UNCOIL DIVA' nan
 'HAIL MAR

Data written successfully!
